# Instrumental Variables

## Part 1 — IV example on mock dataset

### Constructing the dataset

We create four random series of length $N=100000$:

- $x$: education
- $y$: salary
- $z$: ambition (confounding factor)
- $q$: early smoking (candidate instrument)

The relationships are:

1. $x$ and $z$ cause $y$
2. $z$ causes $x$
3. $q$ is correlated with $x$, but **not** with $z$

A problem arises when the confounding factor $z$ is not observed. In that case, we can try to estimate the direct effect of $x$ on $y$ by using $q$ as an **instrument**.

Run the cells below to create the mock dataset.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
N = 100000
ϵ_z = np.random.randn(N)*0.1
ϵ_x = np.random.randn(N)*0.1
ϵ_q = np.random.randn(N)*0.01
ϵ_y = np.random.randn(N)*0.01

In [ ]:
z = 0.1 + ϵ_z
q = 0.5 + 0.1234*ϵ_x + ϵ_q
# q affects x (so q is correlated with x)
x = 0.1 + z + q + ϵ_x
# The true model: y depends on x (coef 0.9) and z (coef 0.4)
y = 1.0 + 0.9*x + 0.4*z + ϵ_y

In [ ]:
df = pd.DataFrame({
    "x": x,
    "y": y,
    "z": z,
    "q": q
})

### Exploring the data

**Exercise 1:** Display the first rows, summary statistics, and the correlation matrix of the dataframe.

In [ ]:
# TODO: show the first rows

In [ ]:
# TODO: summary statistics

In [ ]:
# TODO: correlation matrix

**Question:** Look at the correlation table. For $q$ to be a valid instrument we need:
- `cor(q, x)` non-zero → **relevance** (the instrument predicts $x$)
- `cor(q, z)` close to zero → **exogeneity** (the instrument does not affect $y$ through $z$)

Are these conditions satisfied?

> *Your answer here*

### OLS Regression

We use `linearmodels` to run regressions. The API is slightly different from `statsmodels`:
- `OLS.from_formula("y ~ x", df)` to define the model
- `.fit()` to estimate
- display the result directly (no `.summary()` needed)

In [ ]:
from linearmodels import OLS, IV2SLS

**Exercise 2:** Regress $y$ on $x$ using OLS. What coefficient do you get on $x$?

Remember: the true model is `y = 1.0 + 0.9*x + 0.4*z`. So the true coefficient on $x$ is **0.9**.

In [ ]:
# TODO: run OLS regression of y on x
# model = OLS.from_formula("y ~ x", df)
# res = model.fit()
# res

**Question:** The coefficient on $x$ is **not** 0.9. Why?

*Hint:* $z$ (ambition) affects both $x$ and $y$ but is missing from the regression. This is called **omitted variable bias**.

> *Your answer here*

**Exercise 3:** Now assume $z$ is known. Add it to the regression: `y ~ x + z`. What happens to the coefficient on $x$?

In [ ]:
# TODO: regress y on x and z

**Expected result:** The coefficient on $x$ should now be ≈ 0.9 and the coefficient on $z$ ≈ 0.4 — matching the true model.

But in practice, we often **cannot observe** $z$. That is why we need instrumental variables.

### Instrumental variable regression

We now pretend $z$ is unobserved. We use $q$ as an instrument.

**Why does this work?** $q$ is correlated with $x$ (relevance) but not with $z$ (exogeneity). It gives us variation in $x$ that is "clean" — not contaminated by $z$.

**Exercise 4:** Run an IV regression using `IV2SLS`.

The syntax for instrumenting $x$ with $q$ is:
```python
"y ~ 1 + [x ~ q]"
```
where `[x ~ q]` means "instrument $x$ with $q$".

In [ ]:
# TODO: run IV2SLS regression
# formula = "y ~ 1 + [x ~ q]"
# mod = IV2SLS.from_formula(formula, df)
# res = mod.fit()
# res

**Question:** What is the IV estimate of the coefficient on $x$? Does it match the true value (0.9)?

This is powerful: we recovered the correct causal effect **without** observing $z$.

> *Your answer here*

---

## Part 2 — Return on Education

We follow the excellent R [tutorial](https://www.econometrics-with-r.org/12-6-exercises-10.html) from the *Econometrics with R* book.

**Goal:** measure the effect of schooling on earnings, while correcting the endogeneity bias by using distance to college as an instrument.

### Loading the data

We download the CollegeDistance dataset from `statsmodels`.

In [ ]:
import statsmodels.api as sm
ds = sm.datasets.get_rdataset("CollegeDistance", "AER")

In [ ]:
# Documentation of the dataset
print(ds.__doc__[:1500])

In [ ]:
df = ds.data  # extract the dataframe

**Exercise 5:** Explore the dataframe: show the first rows, the shape, and summary statistics.

In [ ]:
# TODO: df.head()

In [ ]:
# TODO: print shape and describe

### Creating a binary income variable

The `income` column contains text values (`"high"` or `"low"`). We need a numeric version to use in regressions.

In [ ]:
# Check the unique values
df['income'].unique()

**Exercise 6:** Create a new column `income_binary` that equals 1 if income is `"high"` and 0 otherwise.

*Hint:* `(df['income'] == 'high') * 1.0` converts True/False to 1.0/0.0.

In [ ]:
# TODO: create income_binary

In [ ]:
# Verify: should show counts for 0.0 and 1.0
df['income_binary'].value_counts()

### Histogram of distance to college

**Exercise 7:** Plot a histogram of the `distance` variable. This is the variable we will use as an instrument later.

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# TODO: plot a histogram of df['distance']
# Hint: sns.histplot(df['distance']) or plt.hist(df['distance'])
# Add labels and a title

### Naive OLS regression

We now run $\text{income\_binary} = \beta_0 + \beta_1 \cdot \text{education} + u$.

This is "naive" because education might be **endogenous** — unobserved factors (like ability or motivation) could affect both education and income.

**Exercise 8:** Run the naive regression using `IV2SLS` (it works for OLS too — just don't instrument anything).

*Hint:* `"income_binary ~ 1 + education"`

In [ ]:
from linearmodels import IV2SLS

# TODO: run the naive regression

**Question:** Is the coefficient on education significant? What is the R²? Does education explain a lot of the variation in income?

> *Your answer here*

### Adding control variables

**Exercise 9:** Augment the regression with `gender`, `ethnicity`, `urban`, and `unemp`.

Categorical variables (like `gender`, `ethnicity`) are automatically encoded as dummy variables.

In [ ]:
# Check available columns
df.columns

In [ ]:
# TODO: run regression with education + gender + ethnicity + urban + unemp

**Exercise 10:** Check which values `ethnicity` takes. Then change the reference category to `'other'` using the `C()` syntax.

*Hint:* replace `ethnicity` in the formula with `C(ethnicity, Treatment(reference='other'))`.

In [ ]:
df['ethnicity'].unique()

In [ ]:
# TODO: same regression but with C(ethnicity, Treatment(reference='other'))

**Question:** Did the coefficient on education change much after adding controls? Is the model fit (Adj. R²) better?

> *Your answer here*

### The endogeneity problem

Even though the coefficient on education is significant, there might be **endogeneity**: unobserved factors (like ability, family background) could affect both education *and* income. If so, the OLS estimate is biased.

**Question:** Why might `distance` to college be a valid instrument for education?

Think about:
- **Relevance:** Is distance correlated with education?
- **Exogeneity:** Does distance affect income *only* through education?

> *Your answer here*

### IV regression

**Exercise 11:** Run an IV regression where `distance` instruments `education`.

Keep the same control variables (`gender`, `ethnicity` with reference `'other'`, `urban`, `unemp`).

The syntax for instrumenting is:
```python
"income_binary ~ 1 + [education ~ distance] + gender + C(ethnicity, Treatment(reference='other')) + urban + unemp"
```

See: https://bashtage.github.io/linearmodels/

In [ ]:
# TODO: run the IV regression

**Question:** Compare the IV coefficient on education with the OLS coefficient. Is it larger or smaller? What does this tell us about the direction of the bias?

> *Your answer here*

**Final takeaway:** The IV estimate for return on education is considerably **higher** than the OLS estimate. Using the instrument corrects for endogeneity and reveals a stronger causal effect of education on income.

Compare your results with the R [tutorial](https://www.econometrics-with-r.org/12-6-exercises-10.html).